<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
<a href="http://mng.bz/orYv">Build a Large Language Model From Scratch</a> 책의 보충 코드 by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>코드 저장소: <a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

# 미세조정된 모델 로드 및 사용(Load And Use Finetuned Model)

이 노트북은 [ch06.ipynb](ch06.ipynb)를 통해 6장에서 생성하고 저장한 미세조정된 모델을 로드하는 최소한의 코드를 포함합니다.

In [ ]:
from importlib.metadata import version

pkgs = [
    "tiktoken",    # 토크나이저
    "torch",       # 딥러닝 라이브러리
]
for p in pkgs:
    print(f"{p} version: {version(p)}")

In [ ]:
from pathlib import Path

finetuned_model_path = Path("review_classifier.pth")
if not finetuned_model_path.exists():
    print(
        f"'{finetuned_model_path}'를 찾을 수 없습니다.\n"
        "`ch06.ipynb` 노트북을 실행하여 모델을 미세조정하고 저장하세요."
    )

In [ ]:
from previous_chapters import GPTModel


BASE_CONFIG = {
    "vocab_size": 50257,     # 어휘 크기
    "context_length": 1024,  # 컨텍스트 길이
    "drop_rate": 0.0,        # 드롭아웃 비율
    "qkv_bias": True         # 쿼리-키-값 편향
}

model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

CHOOSE_MODEL = "gpt2-small (124M)"

BASE_CONFIG.update(model_configs[CHOOSE_MODEL])

# 기본 모델 초기화
model = GPTModel(BASE_CONFIG)

In [ ]:
import torch

# ch06.ipynb의 6.5절과 같이 모델을 분류기로 변환
num_classes = 2
model.out_head = torch.nn.Linear(in_features=BASE_CONFIG["emb_dim"], out_features=num_classes)

# 그 다음 사전훈련된 가중치 로드
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.load_state_dict(torch.load("review_classifier.pth", map_location=device, weights_only=True))
model.to(device)
model.eval();

In [ ]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")

In [ ]:
# 이 함수는 ch06.ipynb에서 구현되었습니다
def classify_review(text, model, tokenizer, device, max_length=None, pad_token_id=50256):
    model.eval()

    # 모델에 대한 입력 준비
    input_ids = tokenizer.encode(text)
    supported_context_length = model.pos_emb.weight.shape[0]

    # 너무 긴 시퀀스는 잘라냄
    input_ids = input_ids[:min(max_length, supported_context_length)]

    # 가장 긴 시퀀스로 시퀀스를 패딩
    input_ids += [pad_token_id] * (max_length - len(input_ids))
    input_tensor = torch.tensor(input_ids, device=device).unsqueeze(0) # 배치 차원 추가

    # 모델 추론
    with torch.no_grad():
        logits = model(input_tensor.to(device))[:, -1, :]  # 마지막 출력 토큰의 로짓
    predicted_label = torch.argmax(logits, dim=-1).item()

    # 분류 결과 반환
    return "spam" if predicted_label == 1 else "not spam"

In [ ]:
text_1 = (
    "당신은 당첨자입니다. 특별히 선택되어"
    " $1000 현금 또는 $2000 상품을 받으실 수 있습니다."
)

print(classify_review(
    text_1, model, tokenizer, device, max_length=120
))

In [ ]:
text_2 = (
    "안녕, 오늘 저녁 약속 아직 유효한지"
    " 확인하려고 했어. 알려줘!"
)

print(classify_review(
    text_2, model, tokenizer, device, max_length=120
))